This file is for finding the most efficient batch size for training

From previous files, the lr for head is 0.0017378008365631102 and lr range for whole model is ((1.9054607491852948e-06)/10, (1.9054607491852948e-06)/4)

The no. of epochs for head remains const. at 3 and no. of epochs for whole model is 8 as found in prev. file

In [2]:
import pandas as pd
import regex as re
from fastai.vision.all import *

Using powers of 2 as they are efficient on gpus
Varying batch size for whole model between 16 and 64 
Will use batch size of 8 if 16 will perform significantly better than 32

In [3]:
df = pd.read_csv(r'D:\Traffic\labels_processed.csv')

In [4]:
def label_function(dpath):
    class_name = re.findall(r'(\d+)_.*\.png$', dpath.name)
    class_id = int(class_name[0])
    return df['Names'][class_id]

In [5]:
signs = DataBlock(blocks = (ImageBlock, CategoryBlock),
                  get_items = get_image_files,
                  splitter = RandomSplitter(seed = 42),
                  get_y = label_function,
                  item_tfms = Resize(224),
                  batch_tfms = aug_transforms(size = 224, min_scale = 0.75, do_flip = False))

In [6]:
path = Path(r'D:\Traffic\traffic_Data_processed\DATA')

In [7]:
lr_head = 0.0017378008365631102
lr_whole_model = 1.9054607491852948e-06

In [8]:
set_seed(42, reproducible = True)

In [9]:
dls_3_1 = signs.dataloaders(path, bs = 16)

In [10]:
model_3_1 = vision_learner(dls_3_1, arch = resnet34, metrics = error_rate)
model_3_1.fit_one_cycle(3, lr_head)
model_3_1.unfreeze()
model_3_1.fit_one_cycle(8, lr_max = slice(lr_whole_model/10, lr_whole_model/4))

epoch,train_loss,valid_loss,error_rate,time
0,0.921047,0.198062,0.053012,00:30
1,0.202217,0.061283,0.015663,00:27
2,0.097367,0.032212,0.010843,00:27


epoch,train_loss,valid_loss,error_rate,time
0,0.093889,0.033311,0.012048,00:35
1,0.095070,0.032089,0.013253,00:35
2,0.077318,0.027142,0.010843,00:34
3,0.077829,0.027613,0.007229,00:34
4,0.095080,0.025096,0.007229,00:33
5,0.070404,0.023906,0.003614,00:33
6,0.081013,0.026695,0.008434,00:32
7,0.079836,0.026949,0.010843,00:32


In [11]:
set_seed(42, reproducible = True)

In [12]:
dls_3_2 = signs.dataloaders(path, bs = 32)

In [ ]:
model_3_2 = vision_learner(dls_3_2, arch = resnet34, metrics = error_rate)
model_3_2.fit_one_cycle(3, lr_head)
model_3_2.unfreeze()
model_3_2.fit_one_cycle(8, lr_max = slice(lr_whole_model/10, lr_whole_model/4))

epoch,train_loss,valid_loss,error_rate,time
0,1.398693,0.222642,0.063855,00:22
1,0.345230,0.074267,0.022892,00:21
2,0.149158,0.055094,0.020482,00:29


epoch,train_loss,valid_loss,error_rate,time
0,0.110228,0.055667,0.018072,00:36
1,0.096515,0.057483,0.020482,00:36
2,0.096259,0.053477,0.019277,00:38
3,0.100646,0.050280,0.018072,00:36
4,0.095231,0.046494,0.015663,00:25
5,0.093203,0.048655,0.020482,00:39
6,0.090768,0.047002,0.016867,00:38
7,0.093226,0.048003,0.021687,01:04


In [14]:
set_seed(42, reproducible = True)

In [15]:
dls_3_3 = signs.dataloaders(path, bs = 64)

In [16]:
model_3_3 = vision_learner(dls_3_3, arch = resnet34, metrics = error_rate)
model_3_3.fit_one_cycle(3, lr_head)
model_3_3.unfreeze()
model_3_3.fit_one_cycle(8, lr_max = slice(lr_whole_model/10, lr_whole_model/4))

epoch,train_loss,valid_loss,error_rate,time
0,2.107596,0.336565,0.090361,00:36
1,0.784443,0.104941,0.030120,00:38
2,0.364449,0.086720,0.026506,00:35


epoch,train_loss,valid_loss,error_rate,time
0,0.143284,0.089154,0.027711,00:39
1,0.143578,0.086372,0.028916,00:44
2,0.140680,0.087103,0.026506,00:46
3,0.133635,0.081800,0.022892,00:37
4,0.131117,0.080825,0.028916,00:24
5,0.130442,0.082415,0.026506,00:24
6,0.131147,0.079847,0.024096,00:23
7,0.129535,0.080520,0.022892,00:31


So it is clear from here that bs = 16 is optimum